In [16]:
import os
from pathlib import Path

import plotly.graph_objects as go

from paretodo import InvestigationSpace
from helpers.sampling.sampling_lhs import lhs_sampling
from helpers.problems.digital_biolab_simulator import DigitalBioLabSimulator

from plotly_theme import register_theme

register_theme()

In [17]:
CONSTRAINT_THRESHOLD = 400
CONSTRAINT2_THRESHOLD = 0.75
OBJECTIVE_THRESHOLD = 700
N_SAMPLES = 20000
SEED = 11
LOCAL_FOLDER = "../.local"
FOLDER = f"{LOCAL_FOLDER}/bio_testsets/"
os.makedirs(FOLDER, exist_ok=True)

x_name = "temperature"
y_name = "ph_value"
x_display_name = "Temperature"
y_display_name = "pH Value"

main_output_name = "product_concentration"
constraint_name = "aggregate_concentration"
constraint2_name = "purity"
main_display_name = "Titer"
constraint_display_name = "Aggregate Concentration"
constraint2_display_name = "Purity"

In [18]:
investigation_space = InvestigationSpace.from_json(Path(f"{LOCAL_FOLDER}/bio_is/is_2d.json"))

In [19]:
lhs_samples = lhs_sampling(
    investigation_space=investigation_space,
    n_samples=N_SAMPLES,
    seed=SEED,
)
config = {
    "seed": 11,
    "objective_time": "end_of_batch",
    "visualize": "false",
    "reactor": f"{LOCAL_FOLDER}/dbiolab_defaults/MTP_1.json",
    "mammalian_recipe": f"{LOCAL_FOLDER}/dbiolab_defaults/recipe.json",
}
simulator = DigitalBioLabSimulator(config=config)
experiment_dataset = simulator.run_simulation(lhs_samples, iteration=0)

2026-08-11 08:22:35,301 - root - WARNING - Adding normalization params to a normalized dataset.
2026-08-11 08:22:35,305 - root - WARNING - Adding normalization params to a normalized dataset.


In [20]:
df = experiment_dataset.to_dataframe()
df.write_csv(f"{FOLDER}testset_bio_full.csv")

df_filtered_obj = df.filter(df["product_concentration"] > OBJECTIVE_THRESHOLD)
df_filtered_obj.write_csv(f"{FOLDER}testset_bio_full_objective.csv")

# df_filtered_robust_obj = df.filter(
#     (df["product_concentration"] > OBJECTIVE_THRESHOLD) & (df[x_name] > 4)
# )
# df_filtered_robust_obj.write_csv(f"{FOLDER}testset_bio_robust_objective.csv")

df_filtered_constraint = df.filter(df["aggregate_concentration"] < CONSTRAINT_THRESHOLD)
df_filtered_constraint.write_csv(f"{FOLDER}testset_bio_constraint.csv")

df_filtered_constraint2 = df.filter(df["purity"] > CONSTRAINT2_THRESHOLD)
df_filtered_constraint2.write_csv(f"{FOLDER}testset_bio_constraint2.csv")

df_filtered_const_obj = df.filter(
    (df["product_concentration"] > OBJECTIVE_THRESHOLD)
    & (df["aggregate_concentration"] < CONSTRAINT_THRESHOLD)
)
df_filtered_const_obj.write_csv(f"{FOLDER}testset_bio_constraint_objective.csv")

df_filtered_const2_obj = df.filter(
    (df["product_concentration"] > OBJECTIVE_THRESHOLD)
    & (df["purity"] > CONSTRAINT2_THRESHOLD)
)
df_filtered_const2_obj.write_csv(f"{FOLDER}testset_bio_constraint2_objective.csv")

# df_filtered_const_obj_rob = df.filter(
#     (df["product_concentration"] > OBJECTIVE_THRESHOLD)
#     & (df["aggregate_concentration"] < CONSTRAINT_THRESHOLD)
#     & (df[x_name] > 4)
# )
# df_filtered_const_obj_rob.write_csv(
#     f"{FOLDER}testset_bio_constraint_objective_robust.csv"
# )

In [21]:
min_conc = df[main_output_name].min()
max_conc = df[main_output_name].max()
x_min = df[x_name].min()
x_max = df[x_name].max()
y_min = df[y_name].min()
y_max = df[y_name].max()

In [22]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df[x_name],
        y=df[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df[main_output_name],
            colorscale="Viridis",
            showscale=True,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} Testset",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])
fig.show()

In [23]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df[x_name],
        y=df[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df[constraint2_name],
            colorscale="Viridis",
            showscale=True,
        ),
    )
)
fig.update_layout(
    title=f"{constraint2_display_name} Testset",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])
fig.show()

In [24]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df[x_name],
        y=df[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df[constraint_name],
            colorscale="Viridis",
            showscale=True,
        ),
    )
)
fig.update_layout(
    title=f"{constraint_display_name} Testset",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])
fig.show()

In [25]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_obj[x_name],
        y=df_filtered_obj[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_obj[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} > {OBJECTIVE_THRESHOLD} Testset",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])

In [26]:
# fig = go.Figure()
# fig.add_trace(
#     go.Scatter(
#         x=df_filtered_robust_obj[x_name],
#         y=df_filtered_robust_obj[y_name],
#         mode="markers",
#         marker=dict(
#             size=5,
#             color=df_filtered_robust_obj[main_output_name],
#             colorscale="Viridis",
#             showscale=True,
#             cmin=min_conc,
#             cmax=max_conc,
#         ),
#     )
# )
# fig.update_layout(
#     title=f"Robust {main_display_name} > {OBJECTIVE_THRESHOLD} Testset",
#     xaxis_title=x_display_name,
#     yaxis_title=y_display_name,
#     width=800,
#     height=600,
# )
# fig.update_xaxes(range=[x_min, x_max])
# fig.update_yaxes(range=[y_min, y_max])

In [27]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_constraint[x_name],
        y=df_filtered_constraint[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_constraint[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} with constraint: {constraint_display_name} < {CONSTRAINT_THRESHOLD}",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])

In [28]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_const_obj[x_name],
        y=df_filtered_const_obj[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_const_obj[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} > {OBJECTIVE_THRESHOLD} with constraint: {constraint_display_name} < {CONSTRAINT_THRESHOLD}",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])

In [29]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_const2_obj[x_name],
        y=df_filtered_const2_obj[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_const2_obj[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} > {OBJECTIVE_THRESHOLD} with constraint: {constraint2_display_name} > {CONSTRAINT2_THRESHOLD}",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])

In [30]:
# fig = go.Figure()
# fig.add_trace(
#     go.Scatter(
#         x=df_filtered_const_obj_rob[x_name],
#         y=df_filtered_const_obj_rob[y_name],
#         mode="markers",
#         marker=dict(
#             size=5,
#             color=df_filtered_const_obj_rob[main_output_name],
#             colorscale="Viridis",
#             showscale=True,
#             cmin=min_conc,
#             cmax=max_conc,
#         ),
#     )
# )
# fig.update_layout(
#     title=f"Robust {main_display_name} > {OBJECTIVE_THRESHOLD} with constraint: {constraint_display_name} < {CONSTRAINT_THRESHOLD}",
#     xaxis_title=x_display_name,
#     yaxis_title=y_display_name,
#     width=800,
#     height=600,
# )
# fig.update_xaxes(range=[x_min, x_max])
# fig.update_yaxes(range=[y_min, y_max])